In [27]:
import os, sys, re, h5py
import numpy as np
from lib import NanonisAPI, MLAAPI
from lib import FileFunctions, DataProcessing
import time
import matplotlib.pyplot as plt
import sklearn.gaussian_process as gp
from sklearn.model_selection import train_test_split


file_functions = FileFunctions()
(hw_config, error) = file_functions.load_yaml("./sys/config.yml")

nanonis = NanonisAPI(hw_config = hw_config, message_callback = lambda message, message_type: print(f"{message} [{message_type}]"), status_callback = lambda status_str: print(f"Status: {status_str}"))
#mla = MLAAPI(hw_config = hw_config, message_callback = lambda message, message_type: print(f"{message} [{message_type}]"), status_callback = lambda status_str: print(f"Status: {status_str}"))
#nhw = nanonis.nanonis_hardware
data = DataProcessing()
#nanonis.link()

In [31]:
(jitter_dict, error) = nanonis.jitter_tip({"radius (nm)": .1})
z_values = jitter_dict.get("z_values (nm)")
z_avg = np.average(z_values)
z_var = np.var(z_values, ddof = 1)

In [19]:
xy_coords = np.column_stack((x.reshape(-1), y.reshape(-1)))
z_col = z.reshape(-1, 1)

guess_l = (2., 1.)  # In general, x and y have different scales
bounds_l = ((1e-1, 100.),) * 2  # Same bounds for x and y
guess_n = 1.  # Amount of noise
bounds_n = (1e-20, 10.) # Bounds for noise
kernel = gp.kernels.RBF()

xy_train, xy_test, z_train, z_test = train_test_split(xy_coords, z_col, test_size = 0.95)

gpr = gp.GaussianProcessRegressor(kernel, normalize_y = True)


In [26]:
xy_coords.shape

(400, 2)

In [22]:
(z_fit_col, z_std_dev_col) = gpr.predict(xy_coords, return_std = True)
z_fit = z_fit_col.reshape(x.shape)
z_std_dev = z_std_dev_col.reshape(x.shape)

In [2]:
(grid, error) = nanonis.grid_update()
(lists, error) = nanonis.grids_to_lists(grid, direction = "down")

[x_list, y_list] = [lists.get(key) for key in ["x_list (nm)", "y_list (nm)"]]
list_len = len(x_list)

rng = np.random.default_rng()
random_flat_indices = rng.choice(a = list_len, size = 20, replace = False)
random_coordinates = np.array([[x_list[index], y_list[index]] for index in random_flat_indices])

Status: running
